# Обучение YOLOv8 + экспорт в RKNN для Luckfox Pico Mini

Этот ноутбук сам:
1. проверяет, что включён **GPU**
2. просит загрузить `my_dataset.zip`
3. распаковывает и чинит `data.yaml`
4. обучает YOLOv8n
5. экспортирует в ONNX через форк Rockchip
6. квантизует в INT8 и конвертирует в RKNN
7. скачивает `my_model.rknn` и `labels.txt`

## Обязательно перед стартом

1. Меню **Среда выполнения → Сменить среду выполнения**
2. **Аппаратный ускоритель → GPU → T4**
3. Сохранить
4. Если Colab перезапустил сессию — запускайте ячейки снова сверху вниз

Без GPU обучение либо упадёт, либо будет очень долгим на CPU.

Датасет: zip с папкой `my_dataset/` внутри (`train/`, `valid/`, `data.yaml`).


## 1. Установка зависимостей + проверка GPU


In [ ]:
!pip install -q ultralytics pyyaml
import os
os.environ["WANDB_MODE"] = "disabled"

import torch

print("torch:", torch.__version__)
print("cuda.is_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())

if torch.cuda.is_available():
    DEVICE = 0
    BATCH = 16
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    # CPU-запасной режим (медленно). Лучше включить GPU и Runtime → Перезапустить сеанс.
    DEVICE = "cpu"
    BATCH = 8
    print(
        "\n"
        "============================================================\n"
        " GPU НЕ ВКЛЮЧЁН (torch.cuda.is_available() == False)\n"
        "\n"
        " Сделайте так:\n"
        "  1) Среда выполнения → Сменить среду выполнения\n"
        "  2) Аппаратный ускоритель = GPU (T4)\n"
        "  3) Среда выполнения → Перезапустить сеанс\n"
        "  4) Запустите все ячейки заново сверху вниз\n"
        "\n"
        " Сейчас продолжаем на CPU (это может занять очень долго).\n"
        "============================================================\n"
    )

print("DEVICE =", DEVICE, "BATCH =", BATCH)


## 2. Загрузка датасета

Нажмите ▶ — откроется окно выбора файла. Выберите `my_dataset.zip` с компьютера.


In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile, yaml

%cd /content
!rm -rf /content/my_dataset /content/_ds_tmp

uploaded = files.upload()  # выберите my_dataset.zip
assert uploaded, "Файл не загружен"

zip_name = next(n for n in uploaded if n.lower().endswith(".zip"))
print("Загружен:", zip_name, "байт:", len(uploaded[zip_name]))

tmp = Path("/content/_ds_tmp")
tmp.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(tmp)

candidates = list(tmp.rglob("data.yaml"))
assert candidates, "В архиве нет data.yaml"
ds_root = candidates[0].parent
print("Найден датасет:", ds_root)

DATASET_DIR = Path("/content/my_dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
shutil.copytree(ds_root, DATASET_DIR)

train_img = DATASET_DIR / "train" / "images"
val_img = DATASET_DIR / "valid" / "images"
if not val_img.exists():
    val_img = DATASET_DIR / "val" / "images"
assert train_img.exists(), f"Нет {train_img}"
assert val_img.exists(), f"Нет {val_img}"

with open(DATASET_DIR / "data.yaml", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]
nc = int(data.get("nc") or len(names))
assert names and nc == len(names), f"nc={nc}, names={names}"

data["train"] = str(train_img)
data["val"] = str(val_img)
data["nc"] = nc
data["names"] = names
with open(DATASET_DIR / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

n_train = len(list(train_img.glob("*.*")))
n_val = len(list(val_img.glob("*.*")))
print("data.yaml:")
print((DATASET_DIR / "data.yaml").read_text(encoding="utf-8"))
print(f"Картинок: train={n_train}, valid={n_val}, классы={names}")
assert n_train > 0 and n_val > 0

DATASET_YAML = str(DATASET_DIR / "data.yaml")
CLASS_NAMES = names
print("DATASET_YAML =", DATASET_YAML)


## 3. Обучение


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

# на случай, если ячейку 1 перезапускали отдельно — пересчитаем устройство
if "DEVICE" not in globals() or "BATCH" not in globals():
    DEVICE = 0 if torch.cuda.is_available() else "cpu"
    BATCH = 16 if torch.cuda.is_available() else 8

IMG_SIZE = 640
EPOCHS = 100

print("train device =", DEVICE, "batch =", BATCH, "cuda =", torch.cuda.is_available())

model = YOLO("yolov8n.pt")
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
)

BEST_PT = "/content/runs/detect/train/weights/best.pt"
assert Path(BEST_PT).exists(), BEST_PT
print("Лучшие веса:", BEST_PT)


## 4. Проверка качества


In [ ]:
metrics = model.val(data=DATASET_YAML, split="val", plots=True, device=DEVICE)
print(metrics)


## 5. Экспорт в ONNX (форк Rockchip)

Обычный `model.export(format="onnx")` для RKNN не подходит — нужен форк airockchip.


In [ ]:
from pathlib import Path
import re

%cd /content
!rm -rf /content/ultralytics_yolov8
!git clone --depth 1 https://github.com/airockchip/ultralytics_yolov8
%cd /content/ultralytics_yolov8
!pip install -q onnx==1.16.1 onnxruntime==1.19.2

cfg = Path("ultralytics/cfg/default.yaml")
text = cfg.read_text(encoding="utf-8")
text = re.sub(r"(?m)^model:.*$", f"model: {BEST_PT}", text)
text = re.sub(r"(?m)^imgsz:.*$", f"imgsz: {IMG_SIZE}", text)
cfg.write_text(text, encoding="utf-8")
print("model/imgsz в default.yaml:")
!grep -E "^(model|imgsz):" ultralytics/cfg/default.yaml

!PYTHONPATH=./ python ./ultralytics/engine/exporter.py

ONNX_PATH = "/content/runs/detect/train/weights/best.onnx"
assert Path(ONNX_PATH).exists(), ONNX_PATH
!ls -lh "{ONNX_PATH}"
print("ONNX_PATH =", ONNX_PATH)


## 6. Конвертация ONNX → RKNN (INT8, RV1103)


In [ ]:
from pathlib import Path
import re

%cd /content
!pip install -q "rknn-toolkit2==2.3.2"
from rknn.api import RKNN
print("rknn-toolkit2 OK")

!rm -rf /content/rknn_model_zoo
!git clone --depth 1 https://github.com/airockchip/rknn_model_zoo

IMG_PATH = str(Path(DATASET_YAML).parent / "train" / "images")
files = sorted(
    p for p in Path(IMG_PATH).iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
calib = files[:50] if len(files) > 50 else files
subset = Path("/content/data_subset.txt")
subset.write_text("\n".join(str(p) for p in calib) + "\n", encoding="utf-8")
print("калибровка:", len(calib), "файлов ->", subset)

conv = Path("/content/rknn_model_zoo/examples/yolov8/python/convert.py")
src = conv.read_text(encoding="utf-8")
src = re.sub(
    r"DATASET_PATH\s*=\s*[^\n]+",
    "DATASET_PATH = '/content/data_subset.txt'",
    src,
    count=1,
)
conv.write_text(src, encoding="utf-8")
!grep "^DATASET_PATH" "{conv}"

RKNN_OUT = "/content/my_model.rknn"
%cd /content/rknn_model_zoo/examples/yolov8/python
!python3 convert.py "{ONNX_PATH}" rv1103 i8 "{RKNN_OUT}"
assert Path(RKNN_OUT).exists(), RKNN_OUT
!ls -lh "{RKNN_OUT}" 


## 7. Скачать результаты

Нужны два файла: `my_model.rknn` и `labels.txt`.


In [ ]:
from google.colab import files
from pathlib import Path

labels_path = Path("/content/labels.txt")
labels_path.write_text("\n".join(CLASS_NAMES) + "\n", encoding="utf-8")
print("labels.txt:")
print(labels_path.read_text(encoding="utf-8"))

files.download("/content/my_model.rknn")
files.download(str(labels_path))
print("Готово. Положите оба файла рядом и переходите к docs/06-build-and-deploy.md")
